# Activity 5 — Text Extraction
License plate: read the plate number (OCR) and tell the vehicle type from the plate colors.

Before running, install the packages once:

`pip install opencv-python numpy matplotlib pandas pytesseract`

and install the Tesseract OCR program for Windows (for example `winget install UB-Mannheim.TesseractOCR`). Put the photo `license.jpg` in the same folder as this notebook.

In [ ]:
import os
import shutil
from pathlib import Path
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytesseract

# Tesseract: PATH, TESSERACT_CMD, or the default Windows install
tesseract_cmd = os.environ.get('TESSERACT_CMD') or shutil.which('tesseract')
if not tesseract_cmd:
    windows_path = Path(os.environ.get('ProgramFiles', 'C:/Program Files')) / 'Tesseract-OCR' / 'tesseract.exe'
    tesseract_cmd = str(windows_path) if windows_path.is_file() else None
if not tesseract_cmd:
    raise RuntimeError('Install Tesseract OCR, then add it to PATH or set TESSERACT_CMD.')
pytesseract.pytesseract.tesseract_cmd = tesseract_cmd

# Load image
IMAGE_PATH = 'license.jpg'
image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError("Image not found.")

# Resize to a fixed width so the size thresholds below work for any photo size
new_w = 640
new_h = int(image.shape[0] * new_w / image.shape[1])
image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)

print(f'Tesseract {pytesseract.get_tesseract_version()} - image size: {image.shape[1]}x{image.shape[0]}')

plt.figure(figsize=(8, 7))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title('Original image')
plt.axis('off')
plt.show()

Find the plates: a plate is a big, wide rectangle whose border or background is dark or colored. Mark those pixels, then keep the blobs that are the right size and shape.

In [ ]:
def locate_plates(img):
    """Return the bounding boxes (x, y, w, h) of every plate in the image, top to bottom."""
    H, W = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # dark pixels (black borders and characters) or colored pixels (red, green, blue, yellow, orange)
    mask = ((gray < 150) | (hsv[:, :, 1] > 50)).astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if w >= W * 0.12 and h >= H * 0.06 and 1.5 <= w / h <= 3.2:   # big enough, plate-shaped
            boxes.append([x, y, x + w, y + h])

    # merge boxes that overlap (one plate can break into two pieces)
    merged = True
    while merged:
        merged = False
        for i in range(len(boxes)):
            for j in range(i + 1, len(boxes)):
                a, b = boxes[i], boxes[j]
                if a[0] < b[2] and b[0] < a[2] and a[1] < b[3] and b[1] < a[3]:
                    boxes[i] = [min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3])]
                    boxes.pop(j)
                    merged = True
                    break
            if merged:
                break
    return sorted([(x1, y1, x2 - x1, y2 - y1) for x1, y1, x2, y2 in boxes], key=lambda b: (b[1], b[0]))

plate_boxes = locate_plates(image)
print(f'Plates found: {len(plate_boxes)}')

located = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
for x, y, w, h in plate_boxes:
    cv2.rectangle(located, (x, y), (x + w, y + h), (0, 170, 60), 2)

plt.figure(figsize=(8, 7))
plt.imshow(located)
plt.title('Located plates')
plt.axis('off')
plt.show()

Find the characters: crop the plate edges, Otsu threshold, then keep tall, narrow contours (skips the border, holes and small marks).

In [ ]:
def find_characters(plate):
    """Return the cropped plate, its binary mask, and the sorted character boxes."""
    height, width = plate.shape[:2]
    # cut away the plate border
    crop = plate[int(height * 0.10):int(height * 0.90), int(width * 0.04):int(width * 0.96)]

    gray = cv2.GaussianBlur(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY), (3, 3), 0)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if cv2.countNonZero(binary) < binary.size / 2:   # make the background white
        binary = cv2.bitwise_not(binary)

    h, w = binary.shape
    contours, _ = cv2.findContours(255 - binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for contour in contours:
        x, y, bw, bh = cv2.boundingRect(contour)
        if h * 0.35 <= bh <= h * 0.95 and w * 0.02 <= bw <= w * 0.75:
            boxes.append((x, y, bw, bh))

    # drop a box that sits inside a bigger one (broken pieces of one character)
    boxes = [b for b in boxes
             if not any(o != b and o[2] * o[3] > b[2] * b[3] and
                        o[0] <= b[0] + b[2] / 2 <= o[0] + o[2] for o in boxes)]

    # characters that touch become one wide box: cut it where the ink is thinnest
    if boxes:
        char_h = float(np.median([b[3] for b in boxes]))
        split_boxes = []
        for x, y, bw, bh in boxes:
            n = int(round(bw / (0.55 * char_h)))
            if n < 2:
                split_boxes.append((x, y, bw, bh))
                continue
            ink = (binary[y:y + bh, x:x + bw] == 0).sum(axis=0)
            cuts = [0]
            for i in range(1, n):
                guess = int(i * bw / n)
                lo, hi = max(guess - bw // (3 * n), cuts[-1] + 1), min(guess + bw // (3 * n), bw - 1)
                cuts.append(lo + int(np.argmin(ink[lo:hi])) if hi > lo else guess)
            cuts.append(bw)
            for a, b in zip(cuts[:-1], cuts[1:]):
                split_boxes.append((x + a, y, b - a, bh))
        boxes = split_boxes
    return crop, binary, sorted(boxes)

Classify the plate by color. The background tells white, yellow or orange. If the background is white, the color of the characters decides.

In [ ]:
CATEGORIES = {
    'Private':    '#222222',
    'Government': '#d62828',
    'Protocol':   '#2a9d3f',
    'Diplomatic': '#1d4fb8',
    'For-Hire':   '#f4c20d',
    'OEV':        '#f77f00',
}

def classify_plate(crop, binary, boxes):
    """Return (category, background color name, character color name)."""
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

    char_mask = np.zeros(binary.shape, np.uint8)
    for x, y, w, h in boxes:
        char_mask[y:y + h, x:x + w] = 255
    ink = hsv[(char_mask == 255) & (binary == 0)]      # character pixels
    paper = hsv[binary == 255]                          # plate background pixels
    (bg_h, bg_s, _), (ink_h, ink_s, ink_v) = np.median(paper, 0), np.median(ink, 0)

    # colored background
    if bg_s > 100:
        if 8 <= bg_h < 22:
            return 'OEV', 'orange', 'black'
        if 22 <= bg_h <= 38:
            return 'For-Hire', 'yellow', 'black'

    # white background: the character color decides
    if ink_v < 70 or ink_s < 60:
        ink_name = 'black'
    elif ink_h < 10 or ink_h > 170:
        ink_name = 'red'
    elif 35 <= ink_h <= 85:
        ink_name = 'green'
    elif 95 <= ink_h <= 130:
        ink_name = 'blue'
    else:
        ink_name = 'unknown'
    category = {'black': 'Private', 'red': 'Government', 'green': 'Protocol',
                'blue': 'Diplomatic'}.get(ink_name, 'Unknown')
    return category, 'white', ink_name

Read the text: OCR each character several ways, take the majority vote, then group letters and digits at the widest gap.

In [ ]:
LETTERS = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
DIGITS = '0123456789'

def read_character(binary, box, whitelist):
    """Vote across page modes, padding, and sizes to stabilize one character."""
    x, y, w, h = box
    glyph = binary[y:y + h, x:x + w]
    glyph = cv2.medianBlur(glyph, 3)               # remove the small specks of the worn font
    votes = Counter()
    for size in (100, 140):
        resized = cv2.resize(glyph, None, fx=size / h, fy=size / h, interpolation=cv2.INTER_CUBIC)
        for pad in (20, 35, 50):
            padded = cv2.copyMakeBorder(resized, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=255)
            _, padded = cv2.threshold(padded, 127, 255, cv2.THRESH_BINARY)
            for psm in (8, 10, 13):
                config = f'--psm {psm} -c tessedit_char_whitelist={whitelist}'
                text = pytesseract.image_to_string(padded, config=config).strip()
                if text and text[0] in whitelist:
                    votes[text[0]] += 1
    return votes.most_common(1)[0][0] if votes else '?'

def group_image(binary, boxes):
    """Cut out one group of characters (letters or digits) and clean the worn texture."""
    clean = np.full_like(binary, 255)
    for x, y, w, h in boxes:
        clean[y:y + h, x:x + w] = binary[y:y + h, x:x + w]
    left, right = min(b[0] for b in boxes), max(b[0] + b[2] for b in boxes)
    top, bottom = min(b[1] for b in boxes), max(b[1] + b[3] for b in boxes)
    group = cv2.medianBlur(clean[top:bottom, left:right], 3)
    ink = cv2.morphologyEx(255 - group, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))   # fill the worn spots
    return cv2.copyMakeBorder(255 - ink, 30, 30, 30, 30, cv2.BORDER_CONSTANT, value=255)

def read_group(binary, boxes, whitelist):
    """OCR a whole group at once (several scales and page modes), keep the most common text
    that has exactly one character per box. If nothing fits, read the characters one by one."""
    group = group_image(binary, boxes)
    votes = Counter()
    for scale in (1.0, 1.5, 2.0):
        resized = cv2.resize(group, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
        for psm in (7, 8, 13):
            config = f'--psm {psm} -c tessedit_char_whitelist={whitelist}'
            text = pytesseract.image_to_string(resized, config=config).strip().replace(' ', '')
            if len(text) == len(boxes):
                votes[text] += 1
    if votes:
        return votes.most_common(1)[0][0]
    return ''.join(read_character(binary, b, whitelist) for b in boxes)

def read_plate(binary, boxes, numbers_only=False):
    """Letters come before the widest gap, digits after it. Numbers-only plates are all digits."""
    if numbers_only or len(boxes) < 2:
        return read_group(binary, boxes, DIGITS if numbers_only else LETTERS + DIGITS)
    gaps = [boxes[i + 1][0] - (boxes[i][0] + boxes[i][2]) for i in range(len(boxes) - 1)]
    split = int(np.argmax(gaps)) + 1
    letters = read_group(binary, boxes[:split], LETTERS)
    digits = read_group(binary, boxes[split:], DIGITS)
    return f'{letters} {digits}'

def character_strip(binary, boxes):
    """Selected characters only, on white paper, for display."""
    clean = np.full_like(binary, 255)
    for x, y, w, h in boxes:
        clean[y:y + h, x:x + w] = binary[y:y + h, x:x + w]
    left, right = min(b[0] for b in boxes), max(b[0] + b[2] for b in boxes)
    top, bottom = min(b[1] for b in boxes), max(b[1] + b[3] for b in boxes)
    return cv2.copyMakeBorder(clean[top:bottom, left:right], 20, 20, 20, 20, cv2.BORDER_CONSTANT, value=255)

def recognize_plate(img):
    crop, binary, boxes = find_characters(img)
    if not boxes:
        return {'plate': '[none]', 'category': 'Unknown', 'kind': '-', 'background': '-', 'characters': '-',
                'image': img, 'crop': crop, 'boxes': [], 'ocr_input': binary}
    category, background, ink = classify_plate(crop, binary, boxes)
    ph, pw = img.shape[:2]
    kind = 'Motorcycle' if pw / ph < 2.4 else 'Car'      # motorcycle plates are almost square
    numbers_only = category in ('Diplomatic', 'Protocol')       # these plates have numbers only
    return {'plate': read_plate(binary, boxes, numbers_only), 'category': category, 'kind': kind, 'background': background,
            'characters': ink, 'image': img, 'crop': crop, 'boxes': boxes,
            'ocr_input': character_strip(binary, boxes)}

Run on every plate and show each processing stage.

In [ ]:
results = []
for (x, y, w, h) in plate_boxes:
    results.append(recognize_plate(image[y:y + h, x:x + w]))

fig, axes = plt.subplots(len(results), 3, figsize=(13, 3.1 * len(results)), squeeze=False)
for row, r in zip(axes, results):
    annotated = cv2.cvtColor(r['crop'], cv2.COLOR_BGR2RGB)
    for x, y, w, h in r['boxes']:
        cv2.rectangle(annotated, (x, y), (x + w, y + h), (0, 170, 60), 1)

    row[0].imshow(cv2.cvtColor(r['image'], cv2.COLOR_BGR2RGB)); row[0].set_title('Plate', fontsize=9)
    row[1].imshow(annotated);                                    row[1].set_title('Detected characters', fontsize=9)
    row[2].imshow(r['ocr_input'], cmap='gray');                  row[2].set_title(f"OCR -> {r['plate']}", fontsize=9)
    for ax in row:
        ax.axis('off')
plt.tight_layout()
plt.show()

Final results.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6.5))
for ax in axes.flat:
    ax.axis('off')
for ax, r in zip(axes.flat, results):
    color = CATEGORIES.get(r['category'], '#888888')
    label = r['category'] + (' Motorcycle' if r['kind'] == 'Motorcycle' else '')
    ax.axis('on')
    ax.imshow(cv2.cvtColor(r['image'], cv2.COLOR_BGR2RGB))
    ax.set_title(r['plate'], fontsize=14, pad=8)
    ax.set_xlabel(label, fontsize=10, fontweight='bold', color='white',
                  bbox=dict(boxstyle='round,pad=0.4', fc=color, ec='none'), labelpad=10)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(3)
plt.tight_layout()
plt.show()

summary = pd.DataFrame([{'Plate': i + 1, 'Plate number': r['plate'],
                         'Vehicle type': r['category'] + (' Motorcycle' if r['kind'] == 'Motorcycle' else ''),
                         'Background': r['background'], 'Characters': r['characters']}
                        for i, r in enumerate(results)])
summary

OCR can confuse similar glyphs (for example 0 and 6, or O and Q), so check the plate number against the image. Frontal plates are assumed.